In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.covariance import LedoitWolf
from copulas.multivariate import GaussianMultivariate
from ctgan import CTGAN

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

In [2]:
df_train = pd.read_csv('../../data/processed/X_train_v2.csv')

print(f"Toplam Gözlem Sayısı (N): {len(df_train)}")

pozitif_yanginlar = df_train[df_train['area'] > 0].copy()
print(f"Pozitif Hasarlı Yangın Sayısı: {len(pozitif_yanginlar)}")

q75_siniri = pozitif_yanginlar['area'].quantile(0.75)
elit_mega_yanginlar = pozitif_yanginlar[pozitif_yanginlar['area'] >= q75_siniri].copy()

print(f"Top Quartile Sınırı (Log Area): {q75_siniri:.4f}")
print(f"Laboratuvara Girecek Elit Gözlem Sayısı (N): {len(elit_mega_yanginlar)}")

Toplam Gözlem Sayısı (N): 413
Pozitif Hasarlı Yangın Sayısı: 215
Top Quartile Sınırı (Log Area): 2.7970
Laboratuvara Girecek Elit Gözlem Sayısı (N): 54


In [4]:
cekirdek_kolonlar = [
    'X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 
    'month_sin', 'month_cos', 'area'
]

hedef_veri = elit_mega_yanginlar[cekirdek_kolonlar].astype(float)

D = len(cekirdek_kolonlar)
N = len(hedef_veri)

print("--- BOYUT İNDİRGEME (DIMENSIONALITY REDUCTION) ---")
print(f"Orijinal D (Boyut) Sayısı: {len(df_train.columns)}")
print(f"Çekirdek D (Boyut) Sayısı: {D}")
print(f"Gözlem Sayısı (N): {N}")


--- BOYUT İNDİRGEME (DIMENSIONALITY REDUCTION) ---
Orijinal D (Boyut) Sayısı: 23
Çekirdek D (Boyut) Sayısı: 12
Gözlem Sayısı (N): 54


In [5]:
fiziksel_sinirlar = {}

for col in cekirdek_kolonlar:
    fiziksel_sinirlar[col] = {
        'min': df_train[col].min(),
        'max': df_train[col].max()
    }

print(f"RH (Nem) Sınırları     : {fiziksel_sinirlar['RH']['min']} - {fiziksel_sinirlar['RH']['max']}")
print(f"Temp (Sıcaklık) Sınırları: {fiziksel_sinirlar['temp']['min']} - {fiziksel_sinirlar['temp']['max']}")
print(f"Month_Sin (Mevsim) Sınırı: {fiziksel_sinirlar['month_sin']['min']} - {fiziksel_sinirlar['month_sin']['max']}")

def fiziksel_sinirlara_hapset(df_sentetik):
    df_clipped = df_sentetik.copy()
    for col in cekirdek_kolonlar:
        df_clipped[col] = df_clipped[col].clip(
            lower=fiziksel_sinirlar[col]['min'], 
            upper=fiziksel_sinirlar[col]['max']
        )
    df_clipped['X'] = df_clipped['X'].round()
    df_clipped['Y'] = df_clipped['Y'].round()
    return df_clipped

RH (Nem) Sınırları     : 15 - 100
Temp (Sıcaklık) Sınırları: 2.2 - 33.1
Month_Sin (Mevsim) Sınırı: -1.0 - 1.0


In [6]:
lw = LedoitWolf()
lw.fit(hedef_veri)

mu_lw = hedef_veri.mean().values
sigma_lw = lw.covariance_

np.random.seed(42)
matris_A = np.random.multivariate_normal(mean=mu_lw, cov=sigma_lw, size=100)
df_sentetik_A = pd.DataFrame(matris_A, columns=cekirdek_kolonlar)

df_sentetik_A = fiziksel_sinirlara_hapset(df_sentetik_A)

In [7]:
copula_model = GaussianMultivariate()
copula_model.fit(hedef_veri)

df_sentetik_B = copula_model.sample(100)

df_sentetik_B = fiziksel_sinirlara_hapset(df_sentetik_B)

In [8]:
ctgan_model = CTGAN(epochs=100, verbose=False)
ctgan_model.fit(hedef_veri)

df_sentetik_C = ctgan_model.sample(100)

df_sentetik_C = fiziksel_sinirlara_hapset(df_sentetik_C)

In [9]:
def adversarial_validation(real_data, synthetic_data, name):
    
    syn_sample = synthetic_data.head(len(real_data))
    X_adv = pd.concat([real_data, syn_sample]).reset_index(drop=True)
    
    y_adv = np.array([0] * len(real_data) + [1] * len(syn_sample))
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    preds = cross_val_predict(rf, X_adv, y_adv, cv=5, method='predict_proba')[:, 1]
    
    auc = roc_auc_score(y_adv, preds)
    hata_payi = abs(auc - 0.50)
    
    print(f"{name:.<40} AUC Skoru: {auc:.4f} | Mükemmelliğe Uzaklık: {hata_payi:.4f}")
    return hata_payi

def frobenius_norm_test(real_data, synthetic_data, name):
    syn_sample = synthetic_data.head(len(real_data))
    
    corr_real = np.nan_to_num(real_data.corr().values, nan=0.0)
    corr_synth = np.nan_to_num(syn_sample.corr().values, nan=0.0)
    
    fark_normu = np.linalg.norm(corr_real - corr_synth, ord='fro')
    print(f"{name:.<40} Frobenius Hata Puanı: {fark_normu:.4f}")
    return fark_normu

print("=== 1. Adversarial AUC ===")
hata_A = adversarial_validation(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
hata_B = adversarial_validation(hedef_veri, df_sentetik_B, "Aday B (Gaussian Copula)")
hata_C = adversarial_validation(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

print("\n=== 2. Frobenius Norm ===")
frob_A = frobenius_norm_test(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
frob_B = frobenius_norm_test(hedef_veri, df_sentetik_B, "Aday B (Gaussian Copula)")
frob_C = frobenius_norm_test(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

puanlar = {
    "Aday A (Ledoit-Wolf Gaussian)": hata_A + frob_A,
    "Aday B (Gaussian Copula)": hata_B + frob_B,
    "Aday C (CTGAN)": hata_C + frob_C
}

sampiyon_isim = min(puanlar, key=puanlar.get)
print("\n=== EN İYİ MODEL (En Düşük Toplam Hata) ===")
for algoritma, puan in sorted(puanlar.items(), key=lambda item: item[1]):
    print(f"{algoritma:.<40} Toplam Hata Puanı: {puan:.4f}")

print(f"\n EN İYİ MODEL : {sampiyon_isim.upper()} | Toplam Hata Puanı: {puanlar[sampiyon_isim]:.4f}")

=== 1. Adversarial AUC ===
Aday A (Ledoit-Wolf Gaussian)........... AUC Skoru: 0.9990 | Mükemmelliğe Uzaklık: 0.4990
Aday B (Gaussian Copula)................ AUC Skoru: 0.8851 | Mükemmelliğe Uzaklık: 0.3851
Aday C (CTGAN).......................... AUC Skoru: 0.9973 | Mükemmelliğe Uzaklık: 0.4973

=== 2. Frobenius Norm ===
Aday A (Ledoit-Wolf Gaussian)........... Frobenius Hata Puanı: 3.5372
Aday B (Gaussian Copula)................ Frobenius Hata Puanı: 2.3444
Aday C (CTGAN).......................... Frobenius Hata Puanı: 4.1598

=== EN İYİ MODEL (En Düşük Toplam Hata) ===
Aday B (Gaussian Copula)................ Toplam Hata Puanı: 2.7295
Aday A (Ledoit-Wolf Gaussian)........... Toplam Hata Puanı: 4.0361
Aday C (CTGAN).......................... Toplam Hata Puanı: 4.6570

 EN İYİ MODEL : ADAY B (GAUSSIAN COPULA) | Toplam Hata Puanı: 2.7295


In [10]:
df_adaylar = copula_model.sample(1000)
df_adaylar = fiziksel_sinirlara_hapset(df_adaylar)

X_filtre = pd.concat([hedef_veri, df_adaylar]).reset_index(drop=True)
y_filtre = np.array([0] * len(hedef_veri) + [1] * len(df_adaylar))

rf_dedektif = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
sahte_olasiliklari = cross_val_predict(rf_dedektif, X_filtre, y_filtre, cv=5, method='predict_proba')[:, 1]
sentetik_olasiliklari = sahte_olasiliklari[len(hedef_veri):]

df_adaylar['hata_payi'] = np.abs(sentetik_olasiliklari - 0.50)

df_sentetik_B = df_adaylar.sort_values(by='hata_payi').head(100).drop(columns=['hata_payi']).reset_index(drop=True)

In [11]:
def adversarial_validation(real_data, synthetic_data, name):
    
    syn_sample = synthetic_data.head(len(real_data))
    X_adv = pd.concat([real_data, syn_sample]).reset_index(drop=True)
    
    y_adv = np.array([0] * len(real_data) + [1] * len(syn_sample))
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    preds = cross_val_predict(rf, X_adv, y_adv, cv=5, method='predict_proba')[:, 1]
    
    auc = roc_auc_score(y_adv, preds)
    hata_payi = abs(auc - 0.50)
    
    print(f"{name:.<40} AUC Skoru: {auc:.4f} | Mükemmelliğe Uzaklık: {hata_payi:.4f}")
    return hata_payi

def frobenius_norm_test(real_data, synthetic_data, name):
    syn_sample = synthetic_data.head(len(real_data))
    
    corr_real = np.nan_to_num(real_data.corr().values, nan=0.0)
    corr_synth = np.nan_to_num(syn_sample.corr().values, nan=0.0)
    
    fark_normu = np.linalg.norm(corr_real - corr_synth, ord='fro')
    print(f"{name:.<40} Frobenius Hata Puanı: {fark_normu:.4f}")
    return fark_normu

print("=== 1.  YENİ AUC SKORLARI ===")
hata_A = adversarial_validation(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
hata_B = adversarial_validation(hedef_veri, df_sentetik_B, "Aday B (FİLTRELENMİŞ Copula)")
hata_C = adversarial_validation(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

print("\n=== 2.  (Frobenius Norm) ===")
frob_A = frobenius_norm_test(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
frob_B = frobenius_norm_test(hedef_veri, df_sentetik_B, "Aday B (FİLTRELENMİŞ Copula)")
frob_C = frobenius_norm_test(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

puanlar = {
    "Aday A (Ledoit-Wolf Gaussian)": hata_A + frob_A,
    "Aday B (Filtrelenmiş Copula)": hata_B + frob_B,
    "Aday C (CTGAN)": hata_C + frob_C
}

sampiyon_isim = min(puanlar, key=puanlar.get)
print("\n=== EN İYİ MODEL (En Düşük Toplam Hata) ===")
for algoritma, puan in sorted(puanlar.items(), key=lambda item: item[1]):
    print(f"{algoritma:.<40} Toplam Hata Puanı: {puan:.4f}")

print(f"\n EN İYİ MODEL: {sampiyon_isim.upper()} ")

=== 1.  YENİ AUC SKORLARI ===
Aday A (Ledoit-Wolf Gaussian)........... AUC Skoru: 0.9990 | Mükemmelliğe Uzaklık: 0.4990
Aday B (FİLTRELENMİŞ Copula)............ AUC Skoru: 0.7853 | Mükemmelliğe Uzaklık: 0.2853
Aday C (CTGAN).......................... AUC Skoru: 0.9973 | Mükemmelliğe Uzaklık: 0.4973

=== 2.  (Frobenius Norm) ===
Aday A (Ledoit-Wolf Gaussian)........... Frobenius Hata Puanı: 3.5372
Aday B (FİLTRELENMİŞ Copula)............ Frobenius Hata Puanı: 1.3943
Aday C (CTGAN).......................... Frobenius Hata Puanı: 4.1598

=== EN İYİ MODEL (En Düşük Toplam Hata) ===
Aday B (Filtrelenmiş Copula)............ Toplam Hata Puanı: 1.6797
Aday A (Ledoit-Wolf Gaussian)........... Toplam Hata Puanı: 4.0361
Aday C (CTGAN).......................... Toplam Hata Puanı: 4.6570

 EN İYİ MODEL: ADAY B (FILTRELENMIŞ COPULA) 


In [12]:
df_sentetik_B['rain'] = 0

gun_verileri = elit_mega_yanginlar[['day_sin', 'day_cos', 'is_weekend']].sample(n=len(df_sentetik_B), replace=True, random_state=42).reset_index(drop=True)
df_sentetik_B['day_sin'] = gun_verileri['day_sin']
df_sentetik_B['day_cos'] = gun_verileri['day_cos']
df_sentetik_B['is_weekend'] = gun_verileri['is_weekend']

df_sentetik_B['is_peak_season'] = ((df_sentetik_B['temp'] > 22) & (df_sentetik_B['DC'] > 500)).astype(int)

df_sentetik_B['distance_to_center'] = np.sqrt((df_sentetik_B['X'] - 5)**2 + (df_sentetik_B['Y'] - 5)**2)
df_sentetik_B['dynamic_seasonal_hotspot_distance'] = np.where(
    df_sentetik_B['is_peak_season'] == 1,
    np.sqrt((df_sentetik_B['X'] - 8)**2 + (df_sentetik_B['Y'] - 6)**2),
    np.sqrt((df_sentetik_B['X'] - 6)**2 + (df_sentetik_B['Y'] - 5)**2)
)

df_sentetik_B['moderate_wind_danger'] = ((df_sentetik_B['wind'] >= 3.5) & (df_sentetik_B['wind'] <= 6.0)).astype(int)
df_sentetik_B['hot_and_dry'] = ((df_sentetik_B['temp'] >= 20.0) & (df_sentetik_B['RH'] <= 40)).astype(int)
df_sentetik_B['double_drought'] = ((df_sentetik_B['FFMC'] >= 88.0) & (df_sentetik_B['DC'] >= 500.0)).astype(int)
df_sentetik_B['ISI_x_DC'] = df_sentetik_B['ISI'] * df_sentetik_B['DC']

df_sentetik_B = df_sentetik_B[df_train.columns]

X_train_augmented = pd.concat([df_train, df_sentetik_B]).reset_index(drop=True)

X_train_augmented.to_csv('../../data/processed/X_train_augmented_v2.csv', index=False)

print(f"Toplam Gözlem Sayısı: {len(X_train_augmented)}")

Toplam Gözlem Sayısı: 513
